In [ ]:
import os
import numpy as np
import nibabel as nib

def get_bounding_box(mask):
    """
    Calculate the bounding box of a binary mask.
    Returns x_min, x_max, y_min, y_max, z_min, z_max.
    """
    coords = np.argwhere(mask > 0)
    x_min, y_min, z_min = coords.min(axis=0)
    x_max, y_max, z_max = coords.max(axis=0)
    return x_min, x_max, y_min, y_max, z_min, z_max

# Define the path where the segmentation masks are located
path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work/train/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex'  # Replace with your actual path

# Dictionary of required mask filenames
mask_filenames = {
    'hip_left': 'hip_left.nii.gz',
    'hip_right': 'hip_right.nii.gz',
    'gluteus_maximus_left': 'gluteus_maximus_left.nii.gz',
    'gluteus_maximus_right': 'gluteus_maximus_right.nii.gz',
    'prostate': 'prostate.nii.gz',
    'sacrum': 'sacrum.nii.gz'  # This one might not be present
}

# Load the segmentation masks
hip_left_img = nib.load(os.path.join(path, mask_filenames['hip_left']))
hip_right_img = nib.load(os.path.join(path, mask_filenames['hip_right']))
gluteus_left_img = nib.load(os.path.join(path, mask_filenames['gluteus_maximus_left']))
gluteus_right_img = nib.load(os.path.join(path, mask_filenames['gluteus_maximus_right']))
prostate_img = nib.load(os.path.join(path, mask_filenames['prostate']))

hip_left_mask = hip_left_img.get_fdata()
hip_right_mask = hip_right_img.get_fdata()
gluteus_left_mask = gluteus_left_img.get_fdata()
gluteus_right_mask = gluteus_right_img.get_fdata()
prostate_mask = prostate_img.get_fdata()

# Get bounding boxes for each structure
hip_left_bbox = get_bounding_box(hip_left_mask)
hip_right_bbox = get_bounding_box(hip_right_mask)
gluteus_left_bbox = get_bounding_box(gluteus_left_mask)
gluteus_right_bbox = get_bounding_box(gluteus_right_mask)
prostate_bbox = get_bounding_box(prostate_mask)

# Try to load the sacrum mask if it exists
sacrum_mask_path = os.path.join(path, mask_filenames['sacrum'])
if os.path.exists(sacrum_mask_path):
    sacrum_img = nib.load(sacrum_mask_path)
    sacrum_mask = sacrum_img.get_fdata()
    sacrum_bbox = get_bounding_box(sacrum_mask)
    # Use the top of the sacrum mask as the bottom boundary of the rectum
    z_min = int(np.floor(sacrum_bbox[5]))  # z_max of sacrum
else:
    # Use the top of gluteus maximus muscles as the bottom boundary
    z_min = int(np.floor(max(gluteus_left_bbox[5], gluteus_right_bbox[5])))  # z_max of gluteus muscles

# Define the rectum bounding box based on surrounding structures
# Left and right boundaries (x-axis)
x_min = int(np.floor(hip_left_bbox[1]))  # x_max of left hip
x_max = int(np.ceil(hip_right_bbox[0]))  # x_min of right hip

# Anterior and posterior boundaries (y-axis)
y_min = int(np.floor(min(hip_left_bbox[2], hip_right_bbox[2])))
y_max = int(np.ceil(max(hip_left_bbox[3], hip_right_bbox[3])))

# Superior boundary (z-axis)
z_max = int(np.ceil(prostate_bbox[4]))  # z_min of prostate

# Ensure the bounding box is within image dimensions
nx, ny, nz = hip_left_mask.shape
x_min = np.clip(x_min, 0, nx - 1)
x_max = np.clip(x_max, 0, nx - 1)
y_min = np.clip(y_min, 0, ny - 1)
y_max = np.clip(y_max, 0, ny - 1)
z_min = np.clip(z_min, 0, nz - 1)
z_max = np.clip(z_max, 0, nz - 1)

# Create an empty mask for the rectum
rectum_mask = np.zeros_like(hip_left_mask)

# Set the voxels within the bounding box to 1
rectum_mask[x_min:x_max+1, y_min:y_max+1, z_min:z_max+1] = 1

# Save the rectum mask as a NIfTI image
rectum_img = nib.Nifti1Image(rectum_mask, hip_left_img.affine, hip_left_img.header)
nib.save(rectum_img, os.path.join(path, 'rectum.nii.gz'))

print('done')
